In [7]:
# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

sns.set_style("whitegrid")


In [8]:
# CHANGE PATH if needed
df = pd.read_csv("/Users/deepikabudidhi/Desktop/WESAD_preprocessed.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2498913, 9)


,EDA,ECG,Resp,ACC_x,ACC_y,ACC_z,label,Subject,binary_label
0,2.545204,0.135284,-0.387317,1.160905,-1.437833,-0.480366,0,S2,0
1,2.548315,0.035344,-0.396019,-0.126892,-0.269602,0.321515,0,S2,0
2,2.547668,0.101984,-0.404817,0.766681,1.392992,0.443732,0,S2,0
3,2.545424,0.332634,-0.408760,0.776238,-0.269602,0.493977,0,S2,0
4,2.544068,0.682490,-0.407064,0.686642,-0.146011,0.412499,0,S2,0


In [9]:
label_col = "binary_label"

# Drop subject & label from features
X = df.drop(columns=[label_col, "Subject"], errors="ignore")
y = df[label_col].astype(int)
groups = df["Subject"]   # VERY IMPORTANT

print("Features used:", X.columns.tolist())
print("\nLabel distribution:")
print(y.value_counts())


Features used: ['EDA', 'ECG', 'Resp', 'ACC_x', 'ACC_y', 'ACC_z', 'label']

Label distribution:
binary_label
0    1728431
1     770482
Name: count, dtype: int64


In [10]:
gss = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test  = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

print("Train samples:", X_train.shape)
print("Test samples :", X_test.shape)


Train samples: (1930208, 7)
Test samples : (568705, 7)


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [12]:
import numpy as np
import joblib
joblib.dump(scaler, "wesad_scaler.pkl")   # for wesad


['wesad_scaler.pkl']

In [13]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)


In [14]:
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)


In [15]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=10,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)


In [16]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    C=0.1,
    max_iter=3000,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)

print("✅ Linear SVM done")


✅ Linear SVM done


In [17]:
def evaluate(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1 Score :", f1_score(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))


In [18]:
evaluate("Logistic Regression", y_test, log_pred)
evaluate("SVM (RBF)", y_test, svm_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)



===== Logistic Regression =====
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    414792
           1       1.00      1.00      1.00    153913

    accuracy                           1.00    568705
   macro avg       1.00      1.00      1.00    568705
weighted avg       1.00      1.00      1.00    568705


===== SVM (RBF) =====
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    414792
           1       1.00      1.00      1.00    153913

    accuracy                           1.00    568705
   macro avg       1.00      1.00      1.00    568705
weighted avg       1.00      1.00      1.00    568705


===== Random Forest =====
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Classification Report:
         

In [28]:
import numpy as np
import pandas as pd

WINDOW_SIZE = 128      # 4 seconds @ 32 Hz
STEP_SIZE = 64         # 50% overlap

signal_cols = ['EDA', 'ECG', 'Resp', 'ACC_x', 'ACC_y', 'ACC_z']
def create_windows(df):
    X, y, subjects = [], [], []
    
    for subject in df["Subject"].unique():
        sub_df = df[df["Subject"] == subject]
        sub_df = sub_df.reset_index(drop=True)

        for i in range(0, len(sub_df) - WINDOW_SIZE, STEP_SIZE):
            window = sub_df.iloc[i:i+WINDOW_SIZE]
            
            # Majority label
            label = window["binary_label"].mode()[0]

            features = []
            for col in signal_cols:
                features.extend([
                    window[col].mean(),
                    window[col].std(),
                    window[col].min(),
                    window[col].max()
                ])
            
            X.append(features)
            y.append(label)
            subjects.append(subject)
    
    return np.array(X), np.array(y), np.array(subjects)


In [32]:
X, y, subjects = create_windows(df)

print("Windowed samples:", X.shape)
print("Labels distribution:", np.bincount(y))


Windowed samples: (39024, 24)
Labels distribution: [26987 12037]


In [33]:
print(df.columns.tolist())


['EDA', 'ECG', 'Resp', 'ACC_x', 'ACC_y', 'ACC_z', 'label', 'Subject', 'binary_label']


In [34]:
signal_cols = ['EDA', 'ECG', 'Resp', 'ACC_x', 'ACC_y', 'ACC_z']
signal_cols = [c for c in ALL_SIGNAL_COLS if c in df.columns]

print("✅ Using signal columns:", signal_cols)


✅ Using signal columns: ['EDA', 'ECG', 'Resp', 'ACC_x', 'ACC_y', 'ACC_z']


In [35]:
WINDOW_SIZE = 128   # 4 seconds
STEP_SIZE = 64      # 50% overlap

def create_windows(df, signal_cols):
    X, y, subjects = [], [], []

    for subject in df["Subject"].unique():
        sub_df = df[df["Subject"] == subject].reset_index(drop=True)

        for i in range(0, len(sub_df) - WINDOW_SIZE, STEP_SIZE):
            window = sub_df.iloc[i:i+WINDOW_SIZE]

            # Majority label
            label = window["binary_label"].mode()[0]

            features = ['EDA', 'ECG', 'Resp', 'ACC_x', 'ACC_y', 'ACC_z']
            target = 'label'

            X = df[features]
            y = df[target]

            X.append(features)
            y.append(label)
            subjects.append(subject)

    return np.array(X), np.array(y), np.array(subjects)


In [24]:
X, y, subjects = create_windows(df, signal_cols)

print("✅ Windowed samples:", X.shape)
print("✅ Label distribution:", np.bincount(y))


AttributeError: 'DataFrame' object has no attribute 'append'

In [36]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, subjects))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (30142, 24) Test: (8882, 24)


In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # ✅ CORRECT


In [38]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)


In [39]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(C=0.1, max_iter=3000)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=20,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(name, y_true, y_pred):
    print(f"{name:20s} | Acc: {accuracy_score(y_true,y_pred):.3f} | F1: {f1_score(y_true,y_pred):.3f}")

evaluate("Logistic Regression", y_test, log_pred)
evaluate("Linear SVM", y_test, svm_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)


In [ ]:
def evaluate(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1 Score :", f1_score(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))


In [ ]:
evaluate("Logistic Regression", y_test, log_pred)
evaluate("SVM (RBF)", y_test, svm_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)


In [ ]:
models = ["LogReg", "SVM", "RF", "XGB"]
scores = [
    f1_score(y_test, log_pred),
    f1_score(y_test, svm_pred),
    f1_score(y_test, rf_pred),
    f1_score(y_test, xgb_pred)
]

plt.figure(figsize=(7,4))
plt.bar(models, scores)
plt.ylabel("F1 Score")
plt.title("WESAD Model Comparison")
plt.show()


In [ ]:
import joblib
import numpy as np

joblib.dump(rf_model, "wesad_model.pkl")

np.save("X_wesad.npy", X)
np.save("y_wesad.npy", y)

print("WESAD model and data saved successfully.")


In [ ]:
from joblib import dump

dump(model, "wesad_model.pkl")
dump(scaler, "wesad_scaler.pkl")

print("WESAD model and scaler saved successfully")

In [49]:
print(X.shape)

(39024, 24)
